# BERT Analysis for Movie Recommendation

This notebook implements semantic analysis of movie descriptions using Sentence-BERT.

Deliverables generated:
- `outputs/similarity_matrix.csv`
- `outputs/top_similar_movies.csv`

It also includes a comparison between content-based and behavior-based recommendations.


In [1]:
import sys
!{sys.executable} -m pip install sentence-transformers

  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.13.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached sentence_transformers-5.4.1-py3-none-any.whl (571 kB)
Using cached transformers-5.7.0-py3-none-any.whl (10.5 MB)
Using cached huggingface_hub-1.13.0-py3-none-any.whl (660 kB)
Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl (3.7 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl (277 kB)
Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl (341 kB)
   ---------------------------------------- 0.0/114.


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------- ----- 786.4/914.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 1.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 1.4 MB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.2 MB 1.4 MB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.2 MB 1.4 MB/s eta 0:00:02
   ----------------------- ---------------- 1.3/2.2 MB 1.3 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 1.3 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 1.3 MB/s eta 0:00:01
   ----------------------


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sys
print(sys.executable)

d:\PythonFILES\python.exe


In [6]:
import sys
!{sys.executable} -m pip install -U sentence-transformers transformers torch torchvision accelerate

   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.3 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.3 MB 1.4 MB/s eta 0:00:03
   --------- ------------------------------ 1.0/4.3 MB 1.3 MB/s eta 0:00:03
   --------- ------------------------------ 1.0/4.3 MB 1.3 MB/s eta 0:00:03
   -------------- ------------------------- 1.6/4.3 MB 1.3 MB/s eta 0:00:03
   -------------- ------------------------- 1.6/4.3 MB 1.3 MB/s eta 0:00:03
   ---------------- ----------------------- 1.8/4.3 MB 1.2 MB/s eta 0:00:03
   ------------------- -------------------- 2.1/4.3 MB 1.2 MB/s eta 0:00:02
   --------------------- ------------------ 2.4/4.3 MB 1.3 MB/s eta 0:00:02
   ------------------------ --------------- 2.6/4.3 MB 1.3 MB/s eta 0:00:02
   -------------------------- ------------- 2.9/4.3 MB 1.2 MB/s eta 0:00:02
   -------------------------- ---


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


d:\PythonFILES\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
RAW_DIR = Path(r'F:\faculty\Level 3 S_2\Practical Data Mining\project\movie-recommendation-system\movie-recommendation-system\data\raw')
OUTPUTS_DIR = Path(r'F:\faculty\Level 3 S_2\Practical Data Mining\project\movie-recommendation-system\movie-recommendation-system\outputs')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

def normalize_title(title: str) -> str:
    title = re.sub(r'\\(\\d{4}\\)', '', title)
    title = re.sub(r'[^a-zA-Z0-9\\s]', ' ', title.lower())
    return re.sub(r'\\s+', ' ', title).strip()


## 1) Text preprocessing


In [3]:
tmdb = pd.read_csv(RAW_DIR / 'tmdb_5000_movies.csv')
content_df = (
    tmdb[['title', 'overview']]
    .dropna(subset=['title', 'overview'])
    .drop_duplicates(subset=['title'])
    .reset_index(drop=True)
)
content_df['clean_overview'] = content_df['overview'].astype(str).map(clean_text)
content_df.head()


,title,overview,clean_overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di...",in the 22nd century a paraplegic marine is di...
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",captain barbossa long believed to be dead ha...
2,Spectre,A cryptic message from Bond’s past sends him o...,a cryptic message from bond s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...,following the death of district attorney harve...
4,John Carter,"John Carter is a war-weary, former military ca...",john carter is a war weary former military ca...


## 2) Embedding generation (Sentence-BERT)


In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(
    content_df['clean_overview'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)
embeddings.shape


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\PythonFILES\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Arasc\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/150 [00:00<?, ?it/s]

(4797, 384)

## 3) Similarity computation (cosine)


In [5]:
content_similarity = cosine_similarity(embeddings)

# Save matrix for report (first 500 movies keeps file size practical)
matrix_size = min(500, len(content_df))
matrix_titles = content_df['title'].iloc[:matrix_size]
similarity_matrix_df = pd.DataFrame(
    content_similarity[:matrix_size, :matrix_size],
    index=matrix_titles,
    columns=matrix_titles,
)
similarity_matrix_df.to_csv(OUTPUTS_DIR / 'similarity_matrix.csv', index=True)
similarity_matrix_df.shape


(500, 500)

## 4) Top similar movies (content-based)


In [6]:
top_k = 10
rows = []
titles = content_df['title'].tolist()

for i, movie_title in enumerate(titles):
    sims = content_similarity[i]
    top_indices = np.argsort(sims)[::-1]
    rank = 0
    for j in top_indices:
        if i == j:
            continue
        rank += 1
        rows.append({
            'movie_title': movie_title,
            'similar_movie_title': titles[j],
            'similarity_score': float(sims[j]),
            'rank': rank,
        })
        if rank >= top_k:
            break

top_similar_df = pd.DataFrame(rows)
top_similar_df.to_csv(OUTPUTS_DIR / 'top_similar_movies.csv', index=False)
top_similar_df.head(20)


,movie_title,similar_movie_title,similarity_score,rank
0,Avatar,Alien: Resurrection,0.475206,1
1,Avatar,Aliens,0.453270,2
2,Avatar,The Black Hole,0.443850,3
3,Avatar,Serenity,0.426232,4
4,Avatar,Sphere,0.420612,5
5,Avatar,Supernova,0.405178,6
6,Avatar,Space Battleship Yamato,0.397060,7
7,Avatar,Journey to Saturn,0.394447,8
8,Avatar,After Earth,0.392934,9
9,Avatar,"Ultramarines: A Warhammer 40,000 Movie",0.384144,10


## 5) Comparative analysis: content-based vs behavior-based


In [10]:
movies = pd.read_csv(RAW_DIR / 'movies.csv')
ratings = pd.read_csv(RAW_DIR / 'ratings.csv')

# Implicit behavior matrix from ratings (liked = rating >= 4)
ratings['liked'] = (ratings['rating'] >= 4.0).astype(int)
user_item = ratings.pivot_table(index='userId', columns='movieId', values='liked', fill_value=0)
item_user = user_item.T
behavior_similarity = cosine_similarity(item_user)
behavior_movie_ids = item_user.index.to_numpy()

behavior_titles = movies.set_index('movieId')['title']
behavior_norm = {mid: normalize_title(behavior_titles.get(mid, '')) for mid in behavior_movie_ids}
content_norm_to_idx = {normalize_title(t): i for i, t in enumerate(content_df['title'])}

# Find overlap and choose a strong seed movie with enough behavioral interactions
ratings_count = ratings.groupby('movieId').size().to_dict()
overlap = []
for pos, mid in enumerate(behavior_movie_ids):
    n = behavior_norm.get(mid, '')
    if n in content_norm_to_idx and n:
        overlap.append((pos, mid, n, ratings_count.get(mid, 0)))

seed_list = sorted(overlap, key=lambda x: x[3], reverse=True)
if not seed_list:
    seed_list = [(0, behavior_movie_ids[0], behavior_norm.get(behavior_movie_ids[0], ''), 0)]
seed_pos, seed_mid, seed_norm, _ = seed_list[0]
seed_norm_clean = re.sub(r'\s+', ' ', seed_norm).strip()
seed_norm_no_year = re.sub(r'\b\d{4}\b', '', seed_norm_clean).strip()

seed_content_idx = content_norm_to_idx.get(seed_norm_clean)
if seed_content_idx is None:
    seed_content_idx = content_norm_to_idx.get(seed_norm_no_year)

if seed_content_idx is None:
    for i, title in enumerate(content_df['title']):
        if normalize_title(title) in {seed_norm_clean, seed_norm_no_year}:
            seed_content_idx = i
            break

if seed_content_idx is None:
    raise KeyError(f"Cannot resolve seed_norm {seed_norm!r} to a content_df title")

def top_content(seed_idx, k=5):
    sims = content_similarity[seed_idx]
    ids = np.argsort(sims)[::-1]
    out = []
    for j in ids:
        if j == seed_idx:
            continue
        out.append((content_df.iloc[j]['title'], float(sims[j])))
        if len(out) >= k:
            break
    return out

def top_behavior(seed_pos_idx, k=5):
    sims = behavior_similarity[seed_pos_idx]
    ids = np.argsort(sims)[::-1]
    out = []
    for j in ids:
        if j == seed_pos_idx:
            continue
        mid = behavior_movie_ids[j]
        out.append((behavior_titles.get(mid, f'movieId={mid}'), float(sims[j])))
        if len(out) >= k:
            break
    return out

seed_title_content = content_df.iloc[seed_content_idx]['title']
seed_title_behavior = behavior_titles.get(seed_mid)

content_top = top_content(seed_content_idx, k=5)
behavior_top = top_behavior(seed_pos, k=5)

comparison_df = pd.DataFrame({
    'rank': [1, 2, 3, 4, 5],
    'content_based_recommendation': [x[0] for x in content_top],
    'content_similarity': [x[1] for x in content_top],
    'behavior_based_recommendation': [x[0] for x in behavior_top],
    'behavior_similarity': [x[1] for x in behavior_top],
})

print('Seed (content title):', seed_title_content)
print('Seed (behavior title):', seed_title_behavior)
comparison_df


Seed (content title): Toy Story
Seed (behavior title): Toy Story (1995)


,rank,content_based_recommendation,content_similarity,behavior_based_recommendation,behavior_similarity
0,1,Toy Story 2,0.765391,Toy Story 2 (1999),0.506997
1,2,Toy Story 3,0.762004,Star Wars: Episode IV - A New Hope (1977),0.453772
2,3,Hollywood Ending,0.450101,"Sixth Sense, The (1999)",0.442627
3,4,Radio Days,0.438948,"Lion King, The (1994)",0.438397
4,5,Gremlins 2: The New Batch,0.435264,Jurassic Park (1993),0.438366
